# Импорты

In [ ]:
!git clone https://github.com/evagogua/denoiseML

In [ ]:
!pip install -r "/content/denoiseML/requirements.txt"

In [ ]:
import sys
import os

repo_name = "denoiseML"
os.chdir(f'/content/{repo_name}')

from src.models.denoise_model import (
    DenoiseDataset as DenoiseDatasetModel,
    make_last_subtoken_mask,
    prepare_denoise_dataset_from_json
)

from src.inference.denoiser import (
    denoising,
    predict_with_trainer as predict_with_trainer_denoise,
    evaluate_on_test_set as evaluate_on_test_set_denoise,
    test as denoise_test_data
)
from src.models.trainer import (
    plot_training_curves,
    compute_metrics as compute_metrics_training,
    final_report as final_report_training
)

from src.models.classifier_model import (
    ClassificationDataset,
    prepare_classification_dataset_from_json
)

from src.inference.classifier import (
    predict_with_trainer_seq,
    get_simple_metrics,
    compute_metrics_simple
)

# Обучение

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForSequenceClassification

class_model = AutoModelForSequenceClassification.from_pretrained("FacebookAI/xlm-roberta-base")
class_tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")

## Обучение классификационной модели на чистых данных

In [ ]:
train_class_data, test_class_data = prepare_classification_dataset_from_json('/content/denoiseML/data/clean_data_bctw.json', class_tokenizer)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoModelForTokenClassification
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer, DataCollatorForTokenClassification
# from transformers.optimization import AdamW
from torch.optim import AdamW, Adam

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

class_model = AutoModelForSequenceClassification.from_pretrained("FacebookAI/xlm-roberta-base", num_labels=len(train_class_data.tags_))
optimizer = AdamW(class_model.parameters(), lr=1e-5, weight_decay=0.01)
training_args = TrainingArguments(
                  output_dir="trainer_logs", eval_strategy="steps", save_strategy='steps',
                  logging_strategy="steps", save_total_limit=2,
                  num_train_epochs=4, eval_steps=200, disable_tqdm=False,
                  metric_for_best_model='accuracy',
                  warmup_ratio=0.1,
                  report_to="none"
              )
class_trainer = Trainer(
    model=class_model,
    optimizers=(optimizer, None),
    args=training_args,
    data_collator=DataCollatorWithPadding(tokenizer=class_tokenizer),
    train_dataset=train_class_data,
    eval_dataset=test_class_data,
    compute_metrics=compute_metrics_simple)
class_trainer.train()

In [ ]:
plot_training_curves(class_trainer)

In [ ]:
model_save_path = "class_model"
class_trainer.save_model(model_save_path)

## Классификация шумных данных

In [ ]:
train_class_noise_data, test_class_noise_data = prepare_classification_dataset_from_json('/content/denoiseML/data/noise_data_bctw.json', class_tokenizer)

In [ ]:
import numpy as np
from scipy.special import softmax
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

pred_labels, pred_probs = predict_with_trainer_seq(
    class_trainer,
    test_class_noise_data,
    classes=train_class_noise_data.tags_
)

true_labels = [train_class_noise_data.id_to_label[item["labels"].item()] for item in test_class_noise_data]

metrics = get_simple_metrics(true_labels, pred_labels)

## Классификация очищенных данных

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

model_name = "evagogua/tw_bc_denoise_model"

tw_bc_denoise_tokenizer = AutoTokenizer.from_pretrained(model_name)
tw_bc_denoise_model = AutoModelForTokenClassification.from_pretrained(model_name)
tw_bc_denoise_model_trainer = Trainer(
    model=tw_bc_denoise_model,
    args=TrainingArguments(
        output_dir="tw_bc_denoise_model_trainer",
        report_to="none"
    ),
    data_collator=DataCollatorForTokenClassification(tokenizer=tw_bc_denoise_tokenizer)
)

In [ ]:
test_noise_data = prepare_classification_dataset_from_json('/content/denoiseML/data/noise_data_bctw.json', tw_bc_denoise_tokenizer, raw=True)[1]

In [ ]:
from datasets import Dataset
def denoising(data_samples, tr, tokenizer, tag):
  biganswer = []
  test_dataset = DenoiseDataset(data_samples, tokenizer)
  predictions = predict_with_trainer(tr, test_dataset, classes=test_dataset.tags_)
  for item, probs in zip(data_samples, predictions):
    new_sample = {"text":[token for token, mask in zip(item['text'], probs['labels']) if mask == '0'], 'denoise_labels': item["denoise_labels"], 'classification_labels': item["classification_labels"]}
    masked = [item for item in predictions[0]["labels"] if item != -100]
    biganswer.append(new_sample)
  return biganswer

cleaned_data = denoising(test_noise_data, tw_bc_denoise_model_trainer, tw_bc_denoise_tokenizer, ["N", "0"])

hf_dataset = Dataset.from_list(cleaned_data)

denoised_dataset = ClassificationDataset(
    data=hf_dataset,
    tokenizer=class_tokenizer
)

In [ ]:
import numpy as np
from scipy.special import softmax
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

pred_labels, pred_probs = predict_with_trainer_seq(
    class_trainer,
    denoised_dataset,
    classes=train_class_noise_data.tags_
)

true_labels = [train_class_noise_data.id_to_label[item["labels"].item()] for item in denoised_dataset]

metrics = get_simple_metrics(true_labels, pred_labels)